In [2]:
import asyncio

from langgraph.store.memory import InMemoryStore


async def main():

    # ========================================
    # 1. 创建长期记忆 Store
    # ========================================

    store = InMemoryStore()


    # ========================================
    # 2. 定义 namespace
    # ========================================

    namespace = (
        "users",
        "user_123",
        "memories",
    )


    # ========================================
    # 3. 写入长期记忆
    # ========================================

    await store.aput(
        namespace,
        "food_preference",
        {
            "text": "用户喜欢吃辣，尤其喜欢四川火锅。"
        },
    )

    await store.aput(
        namespace,
        "occupation",
        {
            "text": "用户是一名 Python 工程师。"
        },
    )

    await store.aput(
        namespace,
        "tools",
        {
            "text": "用户喜欢使用 Docker 和 PostgreSQL。"
        },
    )


    print("长期记忆写入完成")


    # ========================================
    # 4. 精确读取
    # ========================================

    memory = await store.aget(
        namespace,
        "food_preference",
    )

    print("\n精确读取：")
    print(memory)


    # ========================================
    # 5. 语义搜索
    # ========================================

    results = await store.asearch(
        namespace,
        query="用户喜欢什么食物？",
        limit=3,
    )


    print("\n语义搜索：")

    for item in results:

        print("Key:", item.key)
        print("Value:", item.value)
        print("Score:", item.score)
        print("---")


if __name__ == "__main__":
    await main()

长期记忆写入完成

精确读取：
Item(namespace=['users', 'user_123', 'memories'], key='food_preference', value={'text': '用户喜欢吃辣，尤其喜欢四川火锅。'}, created_at='2026-09-15T10:42:05.397823+00:00', updated_at='2026-09-15T10:42:05.397824+00:00')

语义搜索：
Key: food_preference
Value: {'text': '用户喜欢吃辣，尤其喜欢四川火锅。'}
Score: None
---
Key: occupation
Value: {'text': '用户是一名 Python 工程师。'}
Score: None
---
Key: tools
Value: {'text': '用户喜欢使用 Docker 和 PostgreSQL。'}
Score: None
---


In [ ]:
import asyncio

from langgraph.graph import StateGraph, START, END
from langgraph.store.memory import InMemoryStore
from langgraph.runtime import Runtime
from langgraph.graph.message import MessagesState
store = InMemoryStore()
async def memory_node(
    state: MessagesState,
    runtime: Runtime,
):
    # 用户 ID
    user_id = "user_001"

    # 用户的长期记忆空间
    namespace = ("users", user_id, "memories")

    # 当前用户的问题
    user_message = state["messages"][-1].content

    # 从长期记忆中进行语义搜索
    memories = await runtime.store.asearch(
        namespace,
        query=user_message,
        limit=3,
    )

    print("找到的长期记忆：")

    for memory in memories:
        print("key:", memory.key)
        print("value:", memory.value)
        print("score:", memory.score)

    print("根据长期记忆回答用户问题：")
    print(f"assistant: {memories[0].value['text']}")
    return {
    }
# =========================
# 3. 创建 Graph
# =========================

builder = StateGraph(MessagesState)

builder.add_node("memory", memory_node)

builder.add_edge(START, "memory")
builder.add_edge("memory", END)

graph = builder.compile(store=store)

async def main():

    namespace = ("users", "user_001", "memories")

    await store.aput(
        namespace,
        "name",
        {
            "text": "李明"
        },
    )
    await store.aput(
        namespace,
        "occupation",
        {
            "text": "Python 工程师"
        },
    )
    await store.aput(
        namespace,
        "food_preference",
        {
            "text": "喜欢吃辣"
        },
    )
    await store.aput(
        namespace,
        "tools",
        {
            "text": "喜欢使用 Docker"
        },
    )

    # =========================
    # 5. 调用 Graph
    # =========================

    result = await graph.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "我喜欢吃什么？"
                }
            ]
        }
    )
await main()

NameError: name 'store' is not defined